<a href="https://colab.research.google.com/github/Shaifali07/LLM-projects/blob/main/QuestionBankMaker.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Installations:

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Imports

In [ ]:
!pip install pypdf
!pip install langchain_community
!pip install langchain
!pip install langchain-text-splitters
!pip install langchain_huggingface
!pip install langchain_chroma
!pip install langchain-groq


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.9 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 47.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━

In [ ]:
from pypdf import PdfReader
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
import re
import glob
from pydantic import BaseModel, Field
from typing import List

LLM setup

In [ ]:
from google.colab import userdata
import os

groq_api_key = userdata.get('GROQ_API_KEY')
os.environ["GROQ_API_KEY"] = groq_api_key

In [ ]:
llm=ChatGroq(
    model_name="llama-3.1-8b-instant",
    temperature=0)

Extracting Questions from previous papers

In [ ]:
folder_path = '/content/drive/My Drive/pdf_papers/'
pdf_files = glob.glob(os.path.join(folder_path, '*.pdf'))
text = []
# file_text=' '
for file_path in pdf_files:
    # Open the PDF file in read-binary mode ('rb')
    with open(file_path, 'rb') as pdf_file_obj:
        # Create a PDF reader object
        reader = PdfReader(pdf_file_obj)
        file_text=' '
        # Extract text from each page in the file

        for page in reader.pages:
            file_text += page.extract_text() or "" # use 'or ""' to handle potentially empty text

        # Append the extracted text for the current file to the list
        text.append(file_text)
        print(f"Read file: {os.path.basename(file_path)}")
# DOC_PATH = "/content/cn 1st int 2024.pdf"
# loader = PyPDFLoader(DOC_PATH)
# pages = loader.load()
# text = "\n".join([p.page_content for p in pages])
print(file_text)

Read file: cn 1st int 2024.pdf
Read file: cn_third_sessional_2024.pdf
Read file: cn_updated_second_sessional_2024.pdf
Read file: cn_ext_paper1.pdf
Read file: cn_rem_paper1..pdf
Read file: CN_1-Sessional_30-12-2025.docx.pdf
Read file: cn_sess2_2026.pdf
Read file: cn_sess3_2026.pdf
 DHARMSINH DESAI UNIVERSITY, NADIAD 
FACULTY OF TECHNOLOGY 
THIRD SESSIONAL 
SUBJECT: (23CE611)  COMPUTER NETWORKS 
Page 1 of 2 
Examination : B.Tech Semester VI Seat No. : 
Date : 17-3-2026 Day :Tuesday 
Time : 12:00 to 1:15 Max. Marks : 36 
 
Q.1  Do as directed. [12] 
CO2 E (a) State whether the following statements are True or False. If false, give the correct 
statement. 
1. SNMP TRAP messages are sent from manager to agent. 
2. SNMP is mainly used for sending emails across networks. 
[2] 
CO2 N (b) An Ethernet MAC sublayer receives 30 bytes of data from the upper layer.  How 
many bytes of padding must be added to the data? Show your calculation. 
[2] 
CO2 A (c) Stations in a pure aloha network send fram

In [ ]:
class Clean_text(BaseModel):
  question: str = Field(description="Question extracted from papers")
  marks: int | None = Field(description="Marks for each question ")
parser_clean_text=JsonOutputParser(pydantic_object=Clean_text)
format_instructions_clean_text = parser_clean_text.get_format_instructions()

In [ ]:
prompt_cleaner= ChatPromptTemplate.from_messages ([
    ("system", "You are an expert at processing university exam papers."),
    ("human",'''
    You are given raw extracted text from an exam paper. The text contains noise such as:

headers (university name, subject, date, etc.)
instructions
question numbers (Q.1, Q.2, etc.)
course outcome tags like CO1, CO2
marks like [2], [6]
formatting issues
"OR" sections
mixed sub-questions like (a), (b), (i), (ii)

Your task is to extract clean, meaningful questions.

Instructions:
Remove all irrelevant text such as:
headers
instructions
page numbers
"Attempt any", "Do as directed", etc.
Extract ONLY actual questions.
keep all sub-questions into the same parent  question:
(a), (b), (c)
(i), (ii), (iii)
numbered lists like 1., 2.
If a question contains multiple logical parts, split them into separate questions.
Remove "OR" and treat alternative questions as independent questions.
Preserve the full meaning of each question.
Extract marks if available, otherwise set marks = null.
Clean formatting:
remove extra spaces
fix broken sentences
Output format (STRICT JSON): as per {format_instructions_clean_text}

[
{{
"question": "clean question text",
"marks": number or null
}}
]

Important:
Do NOT include any explanation.
Do NOT include headers or instructions.
Return ONLY valid JSON.
Ensure each question is complete and readable.

Now process the following text:

{input_text}''')
])

In [ ]:
result=[]
import time
chain=prompt_cleaner | llm|parser_clean_text
for t in text:
    time.sleep(0.2)
    result+=chain.invoke({"input_text":t,"format_instructions_clean_text":format_instructions_clean_text})


In [ ]:
for r in result:
  print(r["question"])
# print(result)

What is sender’s window in case of selective repeat, go back n, and stop and wait protocol if the sequence number is 4 bits long?
How many bit(s) error can be detected and corrected by the octets: 0xFF, 0x47 , 0x00 and 0x95?
The receiver receives code word 110010001. The generator polynomial is x 3 + x + 1. Does the received data have an error? (show the procedure)
A supernet is created by combining some class C blocks. It has a first address of 205.16.32.0 and a supernet mask of 255.255.248.0. How many Class C blocks are there in this supernet? Explain.
An IPv4 packet has arrived with the first 8 bits as shown:  01000111... How many bytes of options are being carried by this packet? Show your calculation.
Consider that Host H1 is connected to Host H2 via router R1 and router R2 as shown in figure below. Assign an IP address for all interfaces at all hosts and routers in the network.
What is error control? Where is it provided and where is it mandatory? What provisions are necessary fo

In [ ]:
from langchain_core.documents import Document

docs = []
data= result

for i, q in enumerate(data):
    content = f"""
    Question: {q['question']}
    marks: {q['marks']}

    """
    doc = Document(
        page_content=content,

    )
    # doc = Document(
    #     page_content=content,
    #     metadata=
    #         "marks": q.get("marks")

    #     }
    # )

    docs.append(doc)
print(docs)

[Document(metadata={}, page_content='\n    Question: What is sender’s window in case of selective repeat, go back n, and stop and wait protocol if the sequence number is 4 bits long?\n    marks: 12\n\n    '), Document(metadata={}, page_content='\n    Question: How many bit(s) error can be detected and corrected by the octets: 0xFF, 0x47 , 0x00 and 0x95?\n    marks: 2\n\n    '), Document(metadata={}, page_content='\n    Question: The receiver receives code word 110010001. The generator polynomial is x 3 + x + 1. Does the received data have an error? (show the procedure)\n    marks: 2\n\n    '), Document(metadata={}, page_content='\n    Question: A supernet is created by combining some class C blocks. It has a first address of 205.16.32.0 and a supernet mask of 255.255.248.0. How many Class C blocks are there in this supernet? Explain.\n    marks: 2\n\n    '), Document(metadata={}, page_content='\n    Question: An IPv4 packet has arrived with the first 8 bits as shown:  01000111... How m

Embeddings

In [ ]:
embedings=HuggingFaceEmbeddings()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
persist_directory="/content/db"
vector_store=Chroma.from_documents(
    documents=docs,
    embedding=embedings,
    persist_directory=persist_directory
)

In [ ]:
total_docs=50
retriever=vector_store.as_retriever(
      search_kwargs={"k":total_docs}
)


In [ ]:
class QuestionItem(BaseModel):
  Question: str = Field(description="Question extracted from {context}")
  topic:str=Field(description="Unit of each question")
  Marks: int = Field(description="Marks for each question derived from {context}")


In [ ]:
class QuestionList(BaseModel):
    questions: List[QuestionItem]
parser=JsonOutputParser(pydantic_object=QuestionList)
format_instructions = parser.get_format_instructions()

In [ ]:
class Topic_Formation(BaseModel):
  Units: str = Field(description="Unit extracted from the syllabus")
  topics: str = Field(description="Key topics form the unit")
parser_topic=JsonOutputParser(pydantic_object=Topic_Formation)
format_instructions_topic = parser_topic.get_format_instructions()

In [ ]:
syllabus='''
[1] INTRODUCTION
Uses of computer Networks, Network Hardware-LAN,MAN,WAN, internetworks. Network Software
Design Issues, interfaces & Services, Connection Oriented & Connectionless services. Service
primitives. Relationship of services to protocols. Reference Models - OSI & TCP/IP, their comparison
& critiques.
[2] THE PHYSICAL LAYER
Transmission Media – magnetic media, twisted pair, baseband & broadband, fiber optics. Wireless
Transmission – radio, microwave, infrared & lightwave. Narrowband ISDN, Broadband ISDN &
ATM. Cellular Radio- Paging systems, cordless telephones, analog & digital telephones.
[3] THE DATA LINK LAYER
DLL Design issues, Error Detection & Correction. Elementary Data link Protocols - Utopia, Stop N
Wait, Automatic Repeat Request. Sliding Window Protocols - 1 bit sliding window, Go Back N,
Selective Repeat Protocols.
[4] MEDIUM ACCESS SUBLAYER
Channel Allocation Problem - Static & Dynamic. Multiple Access protocols - ALOHA, CSMA,
Collision Free Protocols, Limited contention protocols, WDMA protocol, wireless LAN protocols.
IEEE standards 802 for LAN & MAN - 802.2, 802.3, 802.4, 802.6 & related numericals. Bridges -
From 802.x to 802.y, transparent Bridges, Spanning Tree, Source Routing Bridges, remote bridge &
problems. Comparison of 802 bridges, High Speed LANs - FDDI, fast ethernet.
[5] THE NETWORK LAYER
Network layer Design issues. Routing Algorithms. Congestion Control Algorithms - general policies,
congestion prevention policies, traffic shaping, flow specifications, congestion control in VC subnets, choke packets, load shedding, jitter control and congestion control for malfunctioning. The network
layer in the internet - the IP protocol, IP addresses & subnets
[6] THE TRANSPORT LAYER
The Transport Service, Elements of Transport Protocols, The Internet Transport Protocols - TCP
service model, TCP protocol, TCP Segment Header, TCP Connection Management, TCP
Transmission Policy, TCP Congestion Policy. UDP & overview of Socket. Performance Issues -
Performance problems in Computer Networks (case study), Measuring Network Performance (case
study).
[7] THE APPLICATION LAYER
Network Security - Traditional Cryptography, Two Fundamental Cryptographic Principles, Secret-Key
Algorithms, Public- key Algorithms, Authentication protocols, Digital Signatures, Social Issues.,
E-mail (case study), SNMP (case study).'''

In [ ]:
prompt_topic = ChatPromptTemplate.from_messages([
    ("system","You are an academic assistant. Your task is to extract structured units from the syllabus."),
    ("human",'''
Instructions:
1. Identify all units/modules from the syllabus {syllabus}.
2. For each unit, extract:
   - unit_name
   - key topics (list of concepts inside the unit)
3. Clean and organize properly.

Output format instructions  {format_instructions_topic}

'''

)])
chain = prompt_topic | llm |parser_topic
topics = chain.invoke({"syllabus":syllabus,"format_instructions_topic":format_instructions_topic})
print(topics)

{'Units': [{'unit_name': 'INTRODUCTION', 'topics': 'Uses of computer Networks, Network Hardware-LAN,MAN,WAN, internetworks. Network Software Design Issues, interfaces & Services, Connection Oriented & Connectionless services. Service primitives. Relationship of services to protocols. Reference Models - OSI & TCP/IP, their comparison & critiques.'}, {'unit_name': 'THE PHYSICAL LAYER', 'topics': 'Transmission Media – magnetic media, twisted pair, baseband & broadband, fiber optics. Wireless Transmission – radio, microwave, infrared & lightwave. Narrowband ISDN, Broadband ISDN & ATM. Cellular Radio- Paging systems, cordless telephones, analog & digital telephones.'}, {'unit_name': 'THE DATA LINK LAYER', 'topics': 'DLL Design issues, Error Detection & Correction. Elementary Data link Protocols - Utopia, Stop N Wait, Automatic Repeat Request. Sliding Window Protocols - 1 bit sliding window, Go Back N, Selective Repeat Protocols.'}, {'unit_name': 'MEDIUM ACCESS SUBLAYER', 'topics': 'Channel 

In [ ]:
unit_list=[u["unit_name"] for u in topics["Units"]]
print(unit_list)
# # for u in topic["units"]:
# #   print (u,"\n")

In [ ]:
# prompt= ChatPromptTemplate.from_messages([
#     ("system", "You are an expert academic assistant. Your task is to process a list of exam questions."),
#     ("human",'''Generate a university-level question bank form the {context} on the following topic: {topic}

# IInstructions:

# 1. Select questions that are conceptually related to the topic.
#    - Do NOT rely on exact keyword matching.
#    - Include questions that are semantically similar or indirectly related.

# 2. Remove duplicate or very similar questions:
#    - Same concept with different wording
#    - Minor variations in phrasing or numbers

# 3. Keep only one best version of each similar question.

# 4. Preserve the marks for each question.

# 5. Clean and rewrite questions for clarity without changing meaning.

# 6. Ignore:
#    - headers
#    - instructions
#    - unrelated questions

# 7. Be strict: include only questions that truly belong to the topic.

# Generate list of the questions as output format must be  {format_instructions}

# '''
#     )])
# chain = prompt | llm |parser

In [75]:
prompt= ChatPromptTemplate.from_messages([
    ("system", "You are an expert academic assistant. Your task is to classsify a list of exam questions."),
    ("human",'''Classify the question bank form given here{context} according to the units. Units contains key topics. you can classify questions according to units. To consider depth of unit you can use key consepts or topics : {topics}

INPUTS:

1. Units:
Each unit contains:
- unit_name
- key_topics (important concepts)

2. Questions:
Each question includes:
- question text
- marks

---

INSTRUCTIONS:

1. For each question, assign the MOST relevant unit based on:
   - semantic meaning (not exact keyword match)
   - similarity to key_topics

2. A question may not contain exact words from key_topics, but if the concept matches, include it.

3. Remove duplicate or very similar questions:
   - same concept with different wording
   - keep only one best version

4. Ignore:
   - headers
   - instructions
   - incomplete or irrelevant text

5. Preserve the marks exactly as given.

6. Each question must be assigned to ONLY ONE best matching unit.

7. Be strict:
   - do not include unrelated questions
   - do not hallucinate or create new questions

Generate list of the questions as output format must be  {format_instructions}

'''
    )])
chain = prompt | llm |parser

In [77]:
unit_list=[u["unit_name"] for u in topics["Units"]]
for u in unit_list:
  docs = retriever.invoke(u)
context = "\n".join([d.page_content for d in docs])

In [86]:
print(context)


    Question: List the layers of OSI model and write the function of each layer
    marks: 5

    

    Question: Discuss similarities and differences between data link layer and transport layer. If acknowledgement is to be provided only at one layer, at which layer should it be included? Why?
    marks: 5

    

    Question: Consider different activities related to email. Activity 1: Send an email from a mail client to a mail server Activity 2: Download an email from mailbox server to a mail client Activity 3: Checking email in a web browser Which is the application level protocol used in each activity?
    marks: 2

    

    Question: What are two reasons for using layered protocols? What is one possible disadvantage of using layered protocols?
    marks: 2

    

    Question: Which transport layer protocol(s) and network layer protocol(s) are used for video, file transfer, DNS and email?
    marks: 2

    

    Question: Compare and contrast data link layer and transport layer. 

In [79]:
result = chain.invoke({"topics": topics,"context":context,"format_instructions":format_instructions})
print(len(result["questions"]))

29


In [84]:
print(result['questions'])

[{'Question': 'List the layers of OSI model and write the function of each layer', 'topic': 'INTRODUCTION', 'Marks': 5}, {'Question': 'Discuss similarities and differences between data link layer and transport layer. If acknowledgement is to be provided only at one layer, at which layer should it be included? Why?', 'topic': 'THE DATA LINK LAYER', 'Marks': 5}, {'Question': 'Consider different activities related to email. Activity 1: Send an email from a mail client to a mail server Activity 2: Download an email from mailbox server to a mail client Activity 3: Checking email in a web browser Which is the application level protocol used in each activity?', 'topic': 'THE APPLICATION LAYER', 'Marks': 2}, {'Question': 'What are two reasons for using layered protocols? What is one possible disadvantage of using layered protocols?', 'topic': 'INTRODUCTION', 'Marks': 2}, {'Question': 'Which transport layer protocol(s) and network layer protocol(s) are used for video, file transfer, DNS and ema

In [83]:
for r in result["questions"]:
  if r["topic"]=="INTRODUCTION":
    print(r["Question"])

List the layers of OSI model and write the function of each layer
What are two reasons for using layered protocols? What is one possible disadvantage of using layered protocols?
Compare the OSI and TCP/IP Reference Models.


KeyError: 'topic'

In [ ]:
# def read_pdf_content(pdf_file_path):
#   full_text = ""
#   with open(pdf_file_path, 'rb') as file:
#         reader = PdfReader(file)
#         for page in reader.pages:
#             text = page.extract_text()
#             if text:
#                 full_text += text + "\n"
#   return full_text


In [ ]:
# file_path = '/content/cn 1st int 2024.pdf'
# extracted_text = read_pdf_content(file_path)
# print(extracted_text)

In [ ]:
# query="ip"
# result=vector_store.similarity_search(query,k=1)
# for i in range(len(result)):
#   print("source:" ,result[i].metadata.get('source','unknown'))
#   print(result[i].page_content)


In [ ]:
text=file_text
# Step 1: Normalize text
text = re.sub(r'\n+', ' ', text)

# Step 2: Split by question labels (a), (b), (c)
parts = re.split(r'\([a-h]\)', text)

questions = []

for part in parts:
    part = part.strip()
    if not part:
        continue

    # Extract marks
    marks_match = re.search(r'\[(\d+)\]', part)
    marks = int(marks_match.group(1)) if marks_match else None

    # Remove noise
    clean = re.sub(r'\[\d+\]', '', part)  # remove [2]
    clean = re.sub(r'CO\d+\s*[A,N,E,R,U,C]', '', clean)  # remove CO2 A
    clean = clean.strip()


    questions.append({
        "question": clean,
        "marks": marks,

    })

# Print result
for q in questions:
    print(q)

{'question': 'DHARMSINH DESAI UNIVERSITY, NADIAD  FACULTY OF TECHNOLOGY  FIRST SESSIONAL  SUBJECT: COMPUTER NETWORKS  Page 1 of 2   Examination : B.Tech Semester VI Seat No. :  Date : 02/01/2024 Day :Tuesday  Time :  1hr 15 mins Max. Marks : 36      Q.1       Do as directed.', 'marks': 12}
{'question': 'What is sender’s window in case of selective repeat, go back n, and stop and wait  protocol if the sequence number is 4 bits long?', 'marks': 2}
{'question': 'How many bit(s) error can be detected and corrected by the octets: 0xFF, 0x47 , 0x00  and 0x95?', 'marks': 2}
{'question': 'The receiver receives code word 110010001. The generator polynomial is x 3 + x + 1.  Does the received data have an error? (show the procedure)', 'marks': 2}
{'question': 'A supernet is created by combining some class C blocks. It has a first address of  205.16.32.0 and a supernet mask of 255.255.248.0. How many Class C blocks are there  in this supernet? Explain.', 'marks': 2}
{'question': 'An IPv4 packet ha

In [ ]:
# text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
# chunks = text_splitter.split_documents(pages)
# print(chunks[1].page_content)

In [ ]:
text = file_text
text = re.sub(r'\n+', ' ', text)
text = re.sub(r'DHARMSINH DESAI UNIVERSITY.*?Max\. Marks\s*:\s*\d+', '', text, flags=re.DOTALL)
text = re.sub(r'INSTRUCTIONS:.*?Page \d+ of \d+', '', text, flags=re.DOTALL)
text = re.sub(r'CO\d+\s*[A-Z]', '', text)
text = re.sub(r'Attempt Any.*?\.', '', text)
text = re.sub(r'Do as directed.*?\.', '', text)
text = re.sub(r'Answer the following.*?\.', '', text)
text = re.sub(r'OR \.', '', text)
text = re.sub(r'(Q\.\d+\s*\[\d+\])','', text)
parts = re.split(r'\([a-h]\)', text)

questions = []

for part in parts:
    part = part.strip()
    if not part:
        continue

    # Extract marks
    marks_match = re.search(r'\[(\d+)\]', part)
    marks = int(marks_match.group(1)) if marks_match else None

    # Remove noise
    clean = re.sub(r'\[\d+\]', '', part)  # remove [2]
    clean = re.sub(r'CO\d+\s*[A,N,E,R,U,C]', '', clean)  # remove CO2 A
    clean = clean.strip()


    questions.append({
        "question": clean,
        "marks": marks,

    })

# print(text)
for q in questions:
    print(q)

{'question': 'What is sender’s window in case of selective repeat, go back n, and stop and wait  protocol if the sequence number is 4 bits long?', 'marks': 2}
{'question': 'How many bit(s) error can be detected and corrected by the octets: 0xFF, 0x47 , 0x00  and 0x95?', 'marks': 2}
{'question': 'The receiver receives code word 110010001. The generator polynomial is x 3 + x + 1.  Does the received data have an error? (show the procedure)', 'marks': 2}
{'question': 'A supernet is created by combining some class C blocks. It has a first address of  205.16.32.0 and a supernet mask of 255.255.248.0. How many Class C blocks are there  in this supernet? Explain.', 'marks': 2}
{'question': 'An IPv4 packet has arrived with the first 8 bits as shown:  01000111...  How many bytes of options are being carried by this packet? Show your calculation.', 'marks': 2}
{'question': 'Consider that Host H1 is connected to Host H2 via router R1 and router R2 as shown in  figure below. Assign an IP address fo

In [ ]:
import re

text = file_text

# -------------------------------
# Step 1: Clean heavy noise
# -------------------------------
text = re.sub(r'\n+', ' ', text)

# Remove headers
text = re.sub(r'DHARMSINH DESAI UNIVERSITY.*?Max\. Marks\s*:\s*\d+', '', text, flags=re.DOTALL)

# Remove instructions block
text = re.sub(r'INSTRUCTIONS:.*?Page \d+ of \d+', '', text, flags=re.DOTALL)

# Remove CO tags
text = re.sub(r'CO\d+\s*[A-Z]', '', text)

# -------------------------------
# Step 2: Split by Q.1 Q.2
# -------------------------------
main_parts = re.split(r'(Q\.\d+)', text)

questions = []

for i in range(1, len(main_parts), 2):
    q_no = main_parts[i]
    content = main_parts[i+1]

    # Remove section text
    content = re.sub(r'Attempt Any.*?\.', '', content)
    content = re.sub(r'Do as directed.*?\.', '', content)
    content = re.sub(r'Answer the following.*?\.', '', content)

    # -------------------------------
    # Step 3: Split (a)(b)(c) + (i)(ii)
    # -------------------------------
    sub_parts = re.split(r'\([a-z]\)|\(\d+\)|\(\w+\)', content)

    for part in sub_parts:
        part = part.strip()

        # Skip garbage
        if len(part) < 20:
            continue

        # -------------------------------
        # Step 4: Extract marks
        # -------------------------------
        marks_match = re.search(r'\[(\d+)\]', part)
        marks = int(marks_match.group(1)) if marks_match else None

        # -------------------------------
        # Step 5: Clean text
        # -------------------------------
        clean = re.sub(r'\[\d+\]', '', part)
        clean = re.sub(r'\bOR\b', '', clean)
        clean = re.sub(r'Page \d+ of \d+', '', clean)
        clean = clean.strip()

        # Skip unwanted chunks
        if any(x in clean.lower() for x in ["figure", "instruction"]):
            continue

        questions.append({
            "question": clean,
            "marks": marks,
            "main_q": q_no
        })

# -------------------------------
# Output
# -------------------------------
for q in questions:
    print(q)

{'question': 'What is sender’s window in case of selective repeat, go back n, and stop and wait  protocol if the sequence number is 4 bits long?', 'marks': 2, 'main_q': 'Q.1'}
{'question': 'error can be detected and corrected by the octets: 0xFF, 0x47 , 0x00  and 0x95?', 'marks': 2, 'main_q': 'Q.1'}
{'question': 'The receiver receives code word 110010001. The generator polynomial is x 3 + x + 1.  Does the received data have an error? (show the procedure)', 'marks': 2, 'main_q': 'Q.1'}
{'question': 'A supernet is created by combining some class C blocks. It has a first address of  205.16.32.0 and a supernet mask of 255.255.248.0. How many Class C blocks are there  in this supernet? Explain.', 'marks': 2, 'main_q': 'Q.1'}
{'question': 'An IPv4 packet has arrived with the first 8 bits as shown:  01000111...  How many bytes of options are being carried by this packet? Show your calculation.', 'marks': 2, 'main_q': 'Q.1'}
{'question': 'What is error control? Where is it provided and where i

In [ ]:
topic='mac'